# 05 — Avaliação de métricas e conclusões do baseline

**Data Challenge | Desafio 1: Classificação de doenças em folhas**

Este notebook reúne a avaliação por classe e as métricas oficiais do baseline, além das conclusões e dificuldades.

São avaliadas duas condições:
- **Baseline (referência):** modelo treinado sem data augmentation.
- **Baseline + augmentation:** modelo treinado com data augmentation.

A avaliação utiliza o conjunto de teste e os modelos salvos pelo notebook 03.

## 1. Preparação do ambiente e carregamento dos dados

Esta célula executa o notebook 01 para reconstruir o conjunto de teste e obter os nomes e a ordem das classes. Assim, este notebook pode ser executado separadamente, desde que as dependências do projeto estejam instaladas.

In [ ]:
from pathlib import Path
from IPython import get_ipython
from IPython.utils.capture import capture_output

notebook_dir = Path.cwd()
if (notebook_dir / 'notebooks').is_dir():
    notebook_dir = notebook_dir / 'notebooks'

notebook_01 = notebook_dir / '01_data_ingestion_preprocessing.ipynb'
assert notebook_01.exists(), (
    f'Não encontrei {notebook_01}. Abra este notebook a partir da pasta do projeto '
    'ou ajuste o caminho de notebook_01.'
)

with capture_output() as run_01_output:
    get_ipython().run_line_magic('run', f'"{notebook_01}"')

print('Classes:', labels)
print('Imagens no conjunto de teste:', len(dataset['test']))

## 2. Carregamento dos modelos baseline

Os modelos precisam ter sido salvos anteriormente pela seção 9 do notebook 03, na pasta `data/processed/`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow import keras
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

ARTIFACTS_DIR = output_dir

model_paths = {
    'baseline': ARTIFACTS_DIR / 'baseline.keras',
    'baseline_aug': ARTIFACTS_DIR / 'baseline_aug.keras',
}

missing_models = [str(path) for path in model_paths.values() if not path.exists()]
assert not missing_models, (
    'Não encontrei os modelos salvos. Execute primeiro o notebook 03 até a seção 9. '
    'Arquivos ausentes: ' + ', '.join(missing_models)
)

models = {
    'baseline': keras.models.load_model(model_paths['baseline']),
    'baseline_aug': keras.models.load_model(model_paths['baseline_aug']),
}

condition_names = {
    'baseline': 'Baseline (referência)',
    'baseline_aug': 'Baseline + augmentation',
}

print('Modelos carregados com sucesso.')

## 3. Avaliação por classe e métricas oficiais

Além da acurácia geral, esta etapa avalia as duas condições utilizando as três métricas definidas para o desafio: Acurácia Global, F1-Score Macro e Matriz de Confusão.

A avaliação é realizada sobre o conjunto de teste, que não foi utilizado durante o treinamento nem na escolha dos pesos do modelo.

A análise por classe permite identificar quais doenças são classificadas corretamente com maior frequência e quais apresentam maior dificuldade de identificação.

In [ ]:
y_true = np.array(dataset['test']['labels'])
predictions = {}
results = {}

for key, model in models.items():
    y_prob = model.predict(test_ds, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    predictions[key] = y_pred

    accuracy = accuracy_score(y_true, y_pred)
    f1_macro = f1_score(y_true, y_pred, average='macro')
    results[key] = {
        'accuracy': float(accuracy),
        'f1_macro': float(f1_macro),
    }

    print(f"===== {condition_names[key]} =====")
    print(f"Acurácia Global: {accuracy:.4f} ({accuracy * 100:.2f}%)")
    print(f"F1-Score Macro: {f1_macro:.4f}")
    print("\\nRelatório por classe:")
    print(classification_report(
        y_true,
        y_pred,
        target_names=labels,
        digits=4,
        zero_division=0,
    ))

### 3.1 Matriz de confusão

A matriz de confusão permite visualizar a quantidade de classificações corretas e incorretas para cada classe. Os valores da diagonal principal representam os acertos, enquanto os valores fora da diagonal representam as classificações incorretas.

In [ ]:
fig, axes = plt.subplots(1, len(predictions), figsize=(8 * len(predictions), 6))
axes = np.atleast_1d(axes)

for ax, (key, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=ax,
    )
    ax.set_title(f"Matriz de Confusão — {condition_names[key]}")
    ax.set_xlabel('Classe prevista')
    ax.set_ylabel('Classe real')

plt.tight_layout()
figures_dir = project_root / 'reports' / 'figures'
figures_dir.mkdir(parents=True, exist_ok=True)
plt.savefig(
    figures_dir / 'baseline_confusion_matrix.png',
    dpi=150,
    bbox_inches='tight',
)
plt.show()

### 3.2 Interpretação dos resultados

A aplicação de data augmentation melhorou o desempenho geral do modelo, elevando a acurácia de 81,25% para 83,59% e o F1-Score Macro de 0,8093 para 0,8365. A principal melhoria ocorreu na identificação da classe `bean_rust`, cujo recall passou de 0,6279 para 0,7674. Já a classe `angular_leaf_spot` apresentou uma pequena redução no recall, enquanto `healthy` manteve o mesmo desempenho.

As matrizes de confusão mostram que o augmentation reduziu os erros entre `bean_rust` e `angular_leaf_spot`, embora ainda existam dificuldades na diferenciação entre essas duas doenças.

De modo geral, o augmentation contribuiu para melhorar o equilíbrio da classificação entre as três classes.

## 4. Conclusões e dificuldades

**Conclusões:**
- Treinamos do zero uma CNN pequena (23.779 parâmetros) em duas condições. A referência parou por early stopping na época 73 (`val_loss` de 0,4403); a condição com augmentation, na época 117 (melhor época 107, `val_loss` de 0,4008). Nenhuma chegou ao teto de 200 épocas.
- Acurácia de validação e teste: 78,95% e 78,12% (referência); 84,96% e 82,81% (augmentation), muito acima do acaso (33%).
- Há indício de ganho com a augmentation: +6,02 p.p. na validação (≈ 8 imagens) e +4,69 p.p. no teste (≈ 6 imagens), com loss menor nos dois. Não dá para atribuir todo o ganho à augmentation, porque cada condição foi treinada uma vez e a condição com augmentation treinou por mais épocas (117 contra 73).
- Pela regra do 01 (escolher pela validação), a condição com augmentation seria a referência das próximas etapas, de acordo com a EDA.

**Dificuldades:**
- **Quantidade de imagens:** o conjunto de teste possui apenas 128 imagens, o que limita a estabilidade das métricas. Cada imagem representa aproximadamente 0,78 ponto percentual da acurácia.
- **Confusão entre doenças:** mesmo após o augmentation, ainda existem classificações incorretas entre `angular_leaf_spot` e `bean_rust`.
- **Variação das imagens:** diferenças de iluminação, enquadramento e fundos podem influenciar as classificações.
- **Detalhamento das lesões:** o redimensionamento das imagens para 224 × 224 pixels pode prejudicar a identificação de lesões pequenas.
- **Generalização:** os resultados obtidos não garantem o mesmo desempenho em imagens de outras lavouras ou em diferentes condições de captura.